In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

csv_path = Path("/mnt/data/results_hist.csv")
df = pd.read_csv(csv_path)

df.columns = [c.strip() if isinstance(c, str) else c for c in df.columns]
df = df.rename(columns={"": "Unused"})
if "Unused" in df.columns:
    df = df.drop(columns=["Unused"])

percent_cols = ["MR", "50PercMR"]
for col in percent_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.rstrip("%")
        df[col] = pd.to_numeric(df[col], errors="coerce")

numeric_cols = [
    "TraceSize", "ExecTime", "Instr", "Cycles", "IPC", "NumBr", "MispBr",
    "BrPerCyc", "MPKI", "CycWP", "CycWPAvg", "CycWPPKI",
    "50PercInstr", "50PercCycles", "50PercIPC", "50PercNumBr",
    "50PercMispBr", "50PercBrPerCyc", "50PercMispBrPerCyc", "50PercMPKI",
    "50PercCycWP", "50PercCycWPAvg", "50PercCycWPPKI"
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df

In [ ]:
versions = list(df["Version"].dropna().unique())
workloads = list(df["Workload"].dropna().unique())

print("Versions:", versions)
print("Workloads:", workloads)

In [ ]:
metrics = [
    "IPC", "MPKI", "CycWPPKI", "MR",
    "50PercIPC", "50PercMPKI", "50PercCycWPPKI", "50PercMR",
    "Cycles", "MispBr", "ExecTime"
]

for metric in metrics:
    if metric not in df.columns:
        continue

    pivot = df.pivot(index="Version", columns="Workload", values=metric)
    ax = pivot.plot(kind="bar", figsize=(10, 5))
    ax.set_title(f"{metric} by Version and Workload")
    ax.set_xlabel("Version")
    ax.set_ylabel(metric)
    ax.legend(title="Workload")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
overall_metrics = ["IPC", "MPKI", "CycWPPKI", "MR", "ExecTime"]

available = [m for m in overall_metrics if m in df.columns]
avg_df = df.groupby("Version")[available].mean(numeric_only=True)

for metric in available:
    ax = avg_df[metric].sort_values().plot(kind="bar", figsize=(10, 5))
    ax.set_title(f"Average {metric} across workloads")
    ax.set_xlabel("Version")
    ax.set_ylabel(metric)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
for workload in workloads:
    subset = df[df["Workload"] == workload].set_index("Version")

    compare_cols = [c for c in [
        "IPC", "MPKI", "CycWPPKI", "MR",
        "50PercIPC", "50PercMPKI", "50PercCycWPPKI", "50PercMR"
    ] if c in subset.columns]

    for metric in compare_cols:
        ax = subset[metric].sort_values().plot(kind="bar", figsize=(10, 5))
        ax.set_title(f"{metric} for workload: {workload}")
        ax.set_xlabel("Version")
        ax.set_ylabel(metric)
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()

In [ ]:
rank_metrics = ["IPC", "MPKI", "CycWPPKI", "MR", "50PercIPC", "50PercMPKI", "50PercCycWPPKI", "50PercMR"]
rank_metrics = [m for m in rank_metrics if m in df.columns]

for workload in workloads:
    subset = df[df["Workload"] == workload].set_index("Version")

    for metric in rank_metrics:
        ascending = metric not in {"IPC", "50PercIPC"}
        ordered = subset[metric].sort_values(ascending=ascending)

        ax = ordered.plot(kind="bar", figsize=(10, 5))
        ax.set_title(f"Sorted {metric} ranking for workload: {workload}")
        ax.set_xlabel("Version")
        ax.set_ylabel(metric)
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.show()